# Prácticas

Para poder asimilar y afianzar los conocimientos adquiridos, en esta unidad vamos a realizar algunas prácticas guiadas.

## Práctica con AEMET OpenData

**Objetivo:** recuperar datos meteorológicos desde internet escribiendo un programa en Python.

AEMET OpenData es una API REST desarrollada por AEMET que permite consultar y reutilizar información meteorológica y climatológica.

### Paso 1 — Obtener una API key

1. Accede a [AEMET OpenData](https://opendata.aemet.es).
2. Solicita una API key usando un correo electrónico.
3. Confirma el correo recibido.
4. Guarda la API key que AEMET te proporciona.

La clave tendrá una estructura parecida a un token largo, pero en este notebook no incluimos ninguna clave real.

> eyJhbGciOiJIUzI1NiJ9.eyJzdWIiOiJqbWudGVyb2dAYWVtZXQuZXMiLCJqdGkiOiINDRiYmVhMy02NDEyLTQxYW...

### Paso 2 — Identificar el endpoint

Para los datos de observación convencionales de las últimas 24 horas, el endpoint utilizado en el material es:

```text
/api/observacion/convencional/todas
```

La URL base es:

```text
https://opendata.aemet.es/opendata/api/observacion/convencional/todas/
```

### Paso 3 — Primera consulta

Una respuesta correcta de AEMET OpenData contiene normalmente información parecida a:

```json
{
  "descripcion": "exito",
  "estado": 200,
  "datos": "https://opendata.aemet.es/opendata/sh/...",
  "metadatos": "https://opendata.aemet.es/opendata/sh/..."
}
```

El código de estado `200` indica que la consulta ha tenido éxito. El campo `datos` contiene una segunda URL desde la que se descargan los datos.

### Paso 4 — Preparar la consulta desde Python

En lugar de escribir la API key directamente en el código, la recuperamos de una variable de entorno llamada `AEMET_API_KEY`.

El ejemplo usa el paquete `requests` —no `request`— y controla los errores de red y la ausencia de la API key.

In [1]:
import os
import requests

url = "https://opendata.aemet.es/opendata/api/observacion/convencional/todas/"
api_key = os.environ.get("AEMET_API_KEY")

print("URL:", url)

if api_key:
    print("API key encontrada en la variable de entorno AEMET_API_KEY.")
else:
    print("No se ha definido AEMET_API_KEY; la petición real no se realizará.")

URL: https://opendata.aemet.es/opendata/api/observacion/convencional/todas/
No se ha definido AEMET_API_KEY; la petición real no se realizará.


Construimos los parámetros y las cabeceras de la petición.

In [2]:
querystring = {"api_key": api_key} if api_key else {}
headers = {"cache-control": "no-cache"}

print("Parámetros preparados:", list(querystring.keys()))
print("Cabeceras:", headers)

Parámetros preparados: []
Cabeceras: {'cache-control': 'no-cache'}


Ahora realizamos la petición únicamente si existe una API key. Así el notebook se puede ejecutar de principio a fin incluso cuando no se dispone de credenciales.

In [3]:
response = None

if api_key:
    try:
        response = requests.get(
            url,
            headers=headers,
            params=querystring,
            timeout=20,
        )
        print("Código HTTP:", response.status_code)
        print("Respuesta:")
        print(response.text[:1000])
    except requests.RequestException as error:
        print("No se pudo realizar la petición:", error)
else:
    print("Petición omitida porque no hay API key.")

Petición omitida porque no hay API key.


Si la respuesta contiene JSON, podemos convertirla directamente a un diccionario de Python.

In [4]:
data = None

if response is not None:
    try:
        data = response.json()
        print("JSON recibido:")
        print(data)
    except ValueError:
        print("La respuesta no contiene JSON válido.")
else:
    print("No hay una respuesta que convertir a JSON.")

No hay una respuesta que convertir a JSON.


Si `estado == 200`, podemos recuperar la URL indicada en `datos` y descargar el contenido.

El siguiente ejemplo guarda el resultado en un fichero cuyo nombre incluye la fecha y la hora actuales.

In [5]:
from datetime import datetime
from pathlib import Path

if isinstance(data, dict) and data.get("estado") == 200 and data.get("datos"):
    url_datos = data["datos"]

    try:
        respuesta_datos = requests.get(url_datos, timeout=20)
        respuesta_datos.raise_for_status()

        timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
        fich_name = f"observacion_{timestamp}.json"

        Path(fich_name).write_bytes(respuesta_datos.content)

        print("Datos guardados en:", fich_name)
        print("Tamaño:", Path(fich_name).stat().st_size, "bytes")
    except requests.RequestException as error:
        print("No se pudieron descargar los datos:", error)
else:
    estado = data.get("estado") if isinstance(data, dict) else None
    print("No hay datos que descargar. Estado:", estado)

No hay datos que descargar. Estado: None


### Ejercicio: `opendata.py`

Escribe un programa que:

1. Recupere los datos de observación de AEMET OpenData.
2. Guarde los datos en un fichero cuyo nombre contenga al menos la fecha de consulta.
3. Como ampliación, guarda la API key y el directorio de salida en variables de entorno.

En el material original se proporciona el fichero de solución `opendata.py`.

# Práctica de lectura y escritura de ficheros JSON

**Objetivo:** leer un fichero JSON, trabajar con sus variables, seleccionar campos de interés y escribir un nuevo fichero JSON.

Para que el notebook sea autocontenido, crearemos un fichero local `datos_observacion.json` con una estructura de ejemplo.

In [6]:
import json
from pathlib import Path

datos_ejemplo = {
    "nubes": [
        {
            "tipo": "Cirrus",
            "descripcion": "Nubes altas y delgadas en forma de filamentos o hebras.",
            "altura": "Alta",
        },
        {
            "tipo": "Cumulus",
            "descripcion": "Nubes blancas y esponjosas con bordes definidos.",
            "altura": "Media",
        },
        {
            "tipo": "Stratus",
            "descripcion": "Nubes bajas y grises que cubren todo el cielo.",
            "altura": "Baja",
        },
    ],
    "meteoros": [
        {
            "tipo": "Lluvia",
            "descripcion": "Precipitación en forma de gotas de agua líquida.",
            "intensidad": "Moderada",
        },
        {
            "tipo": "Nieve",
            "descripcion": "Precipitación en forma de cristales de hielo.",
            "intensidad": "Débil",
        },
        {
            "tipo": "Granizo",
            "descripcion": "Precipitación en forma de bolas de hielo.",
            "intensidad": "Fuerte",
        },
    ],
}

fichero = Path("datos_observacion.json")
fichero.write_text(
    json.dumps(datos_ejemplo, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Fichero creado:", fichero)
print(fichero.read_text(encoding="utf-8"))

Fichero creado: datos_observacion.json
{
  "nubes": [
    {
      "tipo": "Cirrus",
      "descripcion": "Nubes altas y delgadas en forma de filamentos o hebras.",
      "altura": "Alta"
    },
    {
      "tipo": "Cumulus",
      "descripcion": "Nubes blancas y esponjosas con bordes definidos.",
      "altura": "Media"
    },
    {
      "tipo": "Stratus",
      "descripcion": "Nubes bajas y grises que cubren todo el cielo.",
      "altura": "Baja"
    }
  ],
  "meteoros": [
    {
      "tipo": "Lluvia",
      "descripcion": "Precipitación en forma de gotas de agua líquida.",
      "intensidad": "Moderada"
    },
    {
      "tipo": "Nieve",
      "descripcion": "Precipitación en forma de cristales de hielo.",
      "intensidad": "Débil"
    },
    {
      "tipo": "Granizo",
      "descripcion": "Precipitación en forma de bolas de hielo.",
      "intensidad": "Fuerte"
    }
  ]
}


## Paso 1 — Leer un fichero JSON

Definimos una función que abre el fichero, lo interpreta con `json.load()` y devuelve la estructura resultante.

Como el fichero que acabamos de crear utiliza UTF-8, usamos esa misma codificación al leerlo.

In [7]:
import json

fichero = "datos_observacion.json"

def leer_archivo_json(nombre_fichero):
    with open(nombre_fichero, "r", encoding="utf-8") as file:
        data = json.load(file)
    return data

data = leer_archivo_json(fichero)

print(type(data))
print(data)

<class 'dict'>
{'nubes': [{'tipo': 'Cirrus', 'descripcion': 'Nubes altas y delgadas en forma de filamentos o hebras.', 'altura': 'Alta'}, {'tipo': 'Cumulus', 'descripcion': 'Nubes blancas y esponjosas con bordes definidos.', 'altura': 'Media'}, {'tipo': 'Stratus', 'descripcion': 'Nubes bajas y grises que cubren todo el cielo.', 'altura': 'Baja'}], 'meteoros': [{'tipo': 'Lluvia', 'descripcion': 'Precipitación en forma de gotas de agua líquida.', 'intensidad': 'Moderada'}, {'tipo': 'Nieve', 'descripcion': 'Precipitación en forma de cristales de hielo.', 'intensidad': 'Débil'}, {'tipo': 'Granizo', 'descripcion': 'Precipitación en forma de bolas de hielo.', 'intensidad': 'Fuerte'}]}


## Paso 2 — Simular el nombre del fichero como argumento

En un programa ejecutado desde la terminal se podría usar:

```bash
python leer_observacion.py datos_observacion.json
```

y recuperar el nombre con `sys.argv[1]`.

En un notebook es preferible simular explícitamente esos argumentos para que la celda sea reproducible.

In [8]:
import sys

argv_ejemplo = ["leer_observacion.py", "datos_observacion.json"]
fichero_desde_argumento = argv_ejemplo[1]

print("Argumentos simulados:", argv_ejemplo)
print("Fichero seleccionado:", fichero_desde_argumento)

data = leer_archivo_json(fichero_desde_argumento)
print(data)

Argumentos simulados: ['leer_observacion.py', 'datos_observacion.json']
Fichero seleccionado: datos_observacion.json
{'nubes': [{'tipo': 'Cirrus', 'descripcion': 'Nubes altas y delgadas en forma de filamentos o hebras.', 'altura': 'Alta'}, {'tipo': 'Cumulus', 'descripcion': 'Nubes blancas y esponjosas con bordes definidos.', 'altura': 'Media'}, {'tipo': 'Stratus', 'descripcion': 'Nubes bajas y grises que cubren todo el cielo.', 'altura': 'Baja'}], 'meteoros': [{'tipo': 'Lluvia', 'descripcion': 'Precipitación en forma de gotas de agua líquida.', 'intensidad': 'Moderada'}, {'tipo': 'Nieve', 'descripcion': 'Precipitación en forma de cristales de hielo.', 'intensidad': 'Débil'}, {'tipo': 'Granizo', 'descripcion': 'Precipitación en forma de bolas de hielo.', 'intensidad': 'Fuerte'}]}


## Paso 3 — Explorar los campos disponibles

Creamos una función que recorra el JSON y muestre los campos presentes en cada grupo.

In [9]:
def campos_disponibles(datos):
    resultado = {}

    for grupo, registros in datos.items():
        campos = set()

        for registro in registros:
            campos.update(registro.keys())

        resultado[grupo] = sorted(campos)

    return resultado


campos = campos_disponibles(data)

for grupo, nombres in campos.items():
    print(f"{grupo}: {nombres}")

nubes: ['altura', 'descripcion', 'tipo']
meteoros: ['descripcion', 'intensidad', 'tipo']


Ahora podemos seleccionar algunos campos. Para mantener el ejemplo reproducible no usamos `input()`, sino una lista explícita.

In [10]:
campos_seleccionados = ["tipo", "altura", "intensidad"]

print("Campos seleccionados:", campos_seleccionados)

Campos seleccionados: ['tipo', 'altura', 'intensidad']


Aplicamos la selección únicamente a los campos que existan en cada registro.

In [11]:
def seleccionar_campos(datos, seleccion):
    resultado = {}

    for grupo, registros in datos.items():
        resultado[grupo] = []

        for registro in registros:
            reducido = {
                campo: registro[campo]
                for campo in seleccion
                if campo in registro
            }
            resultado[grupo].append(reducido)

    return resultado


datos_seleccionados = seleccionar_campos(data, campos_seleccionados)

print(json.dumps(datos_seleccionados, ensure_ascii=False, indent=2))

{
  "nubes": [
    {
      "tipo": "Cirrus",
      "altura": "Alta"
    },
    {
      "tipo": "Cumulus",
      "altura": "Media"
    },
    {
      "tipo": "Stratus",
      "altura": "Baja"
    }
  ],
  "meteoros": [
    {
      "tipo": "Lluvia",
      "intensidad": "Moderada"
    },
    {
      "tipo": "Nieve",
      "intensidad": "Débil"
    },
    {
      "tipo": "Granizo",
      "intensidad": "Fuerte"
    }
  ]
}


### Ejercicio

Amplía el programa para:

1. Leer todos los campos disponibles.
2. Mostrar los campos al usuario.
3. Permitir seleccionar algunos de ellos.
4. Imprimir los campos seleccionados.

En el material original se proporciona `seleccion.py` como solución.

## Paso 4 — Campos seleccionados por defecto

En la práctica original se propone que los campos:

```text
lon, lat, alt, idema, fint
```

estén seleccionados siempre por defecto y no puedan eliminarse.

El fichero de ejemplo de este notebook no contiene esos campos meteorológicos, así que mostramos la lógica general con una lista separada.

In [12]:
campos_por_defecto = ["lon", "lat", "alt", "idema", "fint"]
campos_usuario = ["tipo", "altura"]

seleccion_final = campos_por_defecto + [
    campo for campo in campos_usuario
    if campo not in campos_por_defecto
]

print("Campos por defecto:", campos_por_defecto)
print("Campos añadidos por el usuario:", campos_usuario)
print("Selección final:", seleccion_final)

Campos por defecto: ['lon', 'lat', 'alt', 'idema', 'fint']
Campos añadidos por el usuario: ['tipo', 'altura']
Selección final: ['lon', 'lat', 'alt', 'idema', 'fint', 'tipo', 'altura']


En el material original se proporciona `defecto.py` como solución de esta ampliación.

## Paso 5 — Escribir un nuevo JSON

Guardamos los datos seleccionados en un fichero cuyo nombre contiene la fecha y la hora actuales.

In [13]:
from datetime import datetime
from pathlib import Path
import json

timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
fichero_salida = Path(f"observacion_seleccion_{timestamp}.json")

fichero_salida.write_text(
    json.dumps(datos_seleccionados, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Fichero de salida:", fichero_salida)
print(fichero_salida.read_text(encoding="utf-8"))

Fichero de salida: observacion_seleccion_20260827T101915.json
{
  "nubes": [
    {
      "tipo": "Cirrus",
      "altura": "Alta"
    },
    {
      "tipo": "Cumulus",
      "altura": "Media"
    },
    {
      "tipo": "Stratus",
      "altura": "Baja"
    }
  ],
  "meteoros": [
    {
      "tipo": "Lluvia",
      "intensidad": "Moderada"
    },
    {
      "tipo": "Nieve",
      "intensidad": "Débil"
    },
    {
      "tipo": "Granizo",
      "intensidad": "Fuerte"
    }
  ]
}


En el material original se proporciona `observacion.py` como solución.

## Paso 6 — Convertir textos de fecha en objetos `datetime`

Cuando un JSON contiene fechas como texto, podemos convertirlas posteriormente a objetos `datetime`.

El formato exacto debe coincidir con el texto recibido.

In [14]:
from datetime import datetime

fechas_texto = [
    "2026-08-27T08:00:00",
    "2026-08-27T09:00:00",
    "2026-08-27T10:00:00",
]

fechas = [
    datetime.strptime(fecha, "%Y-%m-%dT%H:%M:%S")
    for fecha in fechas_texto
]

for fecha in fechas:
    print(fecha, type(fecha))

2026-08-27 08:00:00 <class 'datetime.datetime'>
2026-08-27 09:00:00 <class 'datetime.datetime'>
2026-08-27 10:00:00 <class 'datetime.datetime'>


En el material original se proporciona `leer_json.py` como solución final.